# Import Library

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from nltk.tokenize import word_tokenize

from gensim.models import FastText

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
# Pilihan:
# 0 = Full Dataset
# 1 = 10k
# 2 = 20k
# 3 = 30k
# 4 = 40k

CORPUS_SIZE = 1

if CORPUS_SIZE == 0:
    DATASET_NAME = "full"
elif CORPUS_SIZE == 1:
    DATASET_NAME = "10k"
elif CORPUS_SIZE == 2:
    DATASET_NAME = "20k"
elif CORPUS_SIZE == 3:
    DATASET_NAME = "30k"
elif CORPUS_SIZE == 4:
    DATASET_NAME = "40k"
else:
    raise ValueError(
        "CORPUS_SIZE harus 0, 1, 2, 3, atau 4"
    )

DATA_DIR = Path(
    f"dataset/preprocessed/{DATASET_NAME}"
)

print(f"Corpus : {DATASET_NAME}")
print(f"Path   : {DATA_DIR}")

# Load Dataset & Label Encoder

Dataset

In [ ]:
X_train = pd.read_csv(f"dataset/preprocessed/{CORPUS_SIZE}/X_train.csv").squeeze()
X_test = pd.read_csv(f"dataset/preprocessed//{CORPUS_SIZE}/X_test.csv").squeeze()

y_train = pd.read_csv(f"dataset/preprocessed//{CORPUS_SIZE}/y_train.csv").squeeze()
y_test = pd.read_csv(f"dataset/preprocessed//{CORPUS_SIZE}/y_test.csv").squeeze()

Label Encoder

In [ ]:
le = joblib.load("preprocessed/label_encoder.pkl")
class_names = list(le.classes_)

print(class_names)

# Tokenization

In [ ]:
X_train_tok = [
    word_tokenize(text)
    for text in X_train
]

X_test_tok = [
    word_tokenize(text)
    for text in X_test
]

# Train FastText

In [ ]:
ft_model = FastText(
    vector_size=300,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

ft_model.build_vocab(
    corpus_iterable=X_train_tok
)

ft_model.train(
    corpus_iterable=X_train_tok,
    total_examples=len(X_train_tok),
    epochs=20
)

In [ ]:
os.makedirs("vectorizers/skenario 3", exist_ok=True)
joblib.dump(
    ft_model,f"vectorizers/skenario 3/fasttext_model_{CORPUS_SIZE}.joblib"
)

# Sentence Embedding

In [ ]:
def sentence_vector_fasttext(
    tokens,
    model,
    vector_size=300
):

    vectors = []

    for token in tokens:

        if token in model.wv:

            vectors.append(
                model.wv[token]
            )

    if len(vectors) == 0:

        return np.zeros(
            vector_size
        )

    return np.mean(
        vectors,
        axis=0
    )

# Vectorization

In [ ]:
X_train_ft = np.array([
    sentence_vector_fasttext(
        tokens,
        ft_model
    )
    for tokens in X_train_tok
])

X_test_ft = np.array([
    sentence_vector_fasttext(
        tokens,
        ft_model
    )
    for tokens in X_test_tok
])

# Modeling

Training

In [ ]:
svm_model = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    random_state=42
)

svm_model.fit(
    X_train_ft,
    y_train
)

Predict

In [ ]:
y_pred = svm_model.predict(
    X_test_ft
)

In [ ]:
os.makedirs("models/skenario 3", exist_ok=True)

joblib.dump(
    svm_model,
    f"models/skenario 3/fasttext_svm_{CORPUS_SIZE}.joblib"
)


# Evaluation

## Confussion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(
    figsize=(6,5)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "Confusion Matrix - FastText + SVM"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.tight_layout()

plt.show()

## Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        zero_division=0
    )
)

## Overall

In [ ]:
acc = accuracy_score(
    y_test,
    y_pred
)

prec = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

rec = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(
    "\n=== FastText + SVM ==="
)

print(
    f"Accuracy : {acc:.4f}"
)

print(
    f"Precision: {prec:.4f}"
)

print(
    f"Recall   : {rec:.4f}"
)

print(
    f"F1-Score : {f1:.4f}"
)

In [ ]:
result = pd.DataFrame([
    {
        "Train_Size": len(X_train),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    }
])

os.makedirs(
    "results/traning",
    exist_ok=True
)

RESULT_FILE = "results/traning/fasttext_svm_results.csv"

if os.path.exists(
    RESULT_FILE
):

    previous = pd.read_csv(
        RESULT_FILE
    )

    result = pd.concat(
        [previous, result],
        ignore_index=True
    )

    result = result.drop_duplicates(
        subset=["Train_Size"],
        keep="last"
    )

result = result.sort_values(
    by="Train_Size"
)

result.to_csv(
    RESULT_FILE,
    index=False
)

print(result)